# Notebook 05 — Ablation giá Đồng trong đầu vào (Holdout, toàn bộ module)

Notebook này **không huấn luyện lại**. Nó gom các bảng kết quả Holdout đã lưu từ Module1, Huy, Công/TPE, Vĩ và Quân; sau đó ghép đúng cùng model, seed, split và strategy để đo thay đổi khi thêm kênh `Copper` vào đầu vào.

Hai đối chứng hợp lệ:
- `FS02_PAPER_CU` (có Copper) ↔ `FS03_PAPER_EXO` (không Copper).
- `FS04_METALS_CU` (có Copper) ↔ `FS05_METALS_EXO` (không Copper).

> **Cách diễn giải quan trọng:** “không Copper” nghĩa là bỏ kênh Copper khỏi **tensor đầu vào của mạng**. Với mô hình residual, giá `Copper(t-1)` vẫn được dùng để hoàn nguyên mức giá; Hybrid ARIMA-GRU vẫn có nhánh ARIMA trên chuỗi Copper. Vì vậy không được viết rằng toàn bộ hệ thống hoàn toàn không dùng lịch sử Copper.

CNN, Naive, ARIMA baseline độc lập và mọi kết quả walk-forward đều bị loại khỏi notebook này.

Kết quả TPE/SA của Công dùng HPO theo từng bộ biến. Chênh lệch MAE đo tác động của bộ biến trong quy trình có tuning. V21 của Module1 giữ HP trong mỗi cặp. Bảng coverage ghi riêng các cấu hình còn thiếu.

## 1. Nạp và chuẩn hóa kết quả của năm module

In [ ]:
import glob
import json
import warnings
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import wilcoxon

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid', context='notebook')

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'group_benchmark_summary').is_dir()), Path.cwd())

OUT = Path('/kaggle/working/05_copper_input_ablation') if Path('/kaggle/working').exists() else ROOT / '05_copper_input_ablation'
OUT.mkdir(parents=True, exist_ok=True)

def find_one(local_candidates, filename):
    for candidate in local_candidates:
        path = ROOT / candidate
        if path.exists():
            return path
    hits = [Path(p) for p in glob.glob(f'/kaggle/input/**/{filename}', recursive=True)]
    hits += [Path(p) for p in glob.glob(f'**/{filename}', recursive=True) if '05_copper_input_ablation' not in p]
    if not hits:
        raise FileNotFoundError(f'Không tìm thấy {filename}. Hãy chạy notebook nguồn hoặc gắn output vào Kaggle Input.')
    return max(hits, key=lambda p: p.stat().st_mtime)

def numeric_frame(df, cols=('ratio', 'seed', 'mae', 'rmse', 'mape_pct', 'r2')):
    for col in cols:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

def keep_schema(df):
    cols = ['module', 'model', 'objective', 'feature_set', 'ratio', 'scheme', 'seed',
            'mae', 'rmse', 'mape_pct', 'r2', 'source_file', 'protocol_note']
    for col in cols:
        if col not in df:
            df[col] = np.nan
    return numeric_frame(df[cols].copy())

sources = {}
frames = []

In [ ]:
# Module1 — ưu tiên bảng đã được Notebook 04 tái đánh giá từ trọng số.
try:
    p_m1 = find_one([
        '04_evaluation_report/eval_holdout_all_models.csv',
        'Module1/offical-notebooks/04_evaluation_report/eval_holdout_all_models.csv',
    ], 'eval_holdout_all_models.csv')
    m1 = pd.read_csv(p_m1)
    sources['Module1'] = str(p_m1)
except FileNotFoundError:
    parts = []
    for folder, filename in [('02_sa_artifacts', 'sa_holdout_results.csv'),
                             ('03_nonsa_artifacts', 'nonsa_holdout_results.csv')]:
        try:
            path = find_one([f'{folder}/{filename}', f'Module1/offical-notebooks/{folder}/{filename}'], filename)
        except FileNotFoundError:
            continue
        parts.append(pd.read_csv(path).assign(source_file=str(path)))
    archive = '17-11-SA-LSTM-standalone-V21'
    sa = find_one([f'Module1/{folder}/{archive}/copper_sa_lstm_h1_ablation_v21/results_h1_ablation.csv'
                   for folder in ('old-notebooks/results', 'results')], 'results_h1_ablation.csv')
    archive = '17-11-Copper-NonSA-Fair-Full-Protocol-V20'
    ns = find_one([f'Module1/{folder}/{archive}/grid_results.csv'
                   for folder in ('old-notebooks/results', 'results')], 'grid_results.csv')
    legacy = pd.concat([pd.read_csv(sa).assign(source_file=str(sa)),
                        pd.read_csv(ns).assign(source_file=str(ns))], ignore_index=True)
    mapping = {'paper3_with_copper': 'FS02_PAPER_CU', 'paper3_without_copper': 'FS03_PAPER_EXO',
               'legacy14_with_copper': 'FS04_METALS_CU', 'legacy14_without_copper': 'FS05_METALS_EXO',
               'copper_paper3': 'FS02_PAPER_CU', 'copper_legacy14': 'FS04_METALS_CU'}
    legacy = legacy[legacy['feature_set'].isin(mapping)].copy()
    legacy['feature_set'] = legacy['feature_set'].map(mapping)
    # Keep both arms from the same experiment when a newer branch is available.
    current_models = {model for part in parts for model in part['model']}
    parts.append(legacy[~legacy['model'].isin(current_models)])
    m1 = pd.concat(parts, ignore_index=True)
    sources['Module1'] = sorted(m1['source_file'].unique().tolist())
allowed_m1 = {
    'baseline_lstm', 'baseline_rnn', 'paper_sa_lstm', 'reopt_sa_lstm',
    'lstm_direct', 'gru_direct', 'lstm_residual', 'gru_residual', 'ridge_residual',
}
m1 = m1[m1['model'].isin(allowed_m1)].copy()
m1['module'] = 'Khoa — Module1'
m1['objective'] = np.where(m1['model'].str.contains('residual'), 'residual', 'direct')
if 'source_file' not in m1:
    m1['source_file'] = sources['Module1']
m1['protocol_note'] = 'One-shot Holdout; scaler fit train-only'
frames.append(keep_schema(m1))

# Huy — chỉ hai mô hình học máy; loại Naive và ARIMA baseline.
p_huy = find_one(['huy task/result/results_by_seed.csv'], 'results_by_seed.csv')
huy = pd.read_csv(p_huy)
huy = huy[huy['Mô hình'].isin(['GRU Direct', 'Hybrid ARIMA-GRU'])].copy()
huy['module'] = 'Huy — PP5'
huy['model'] = huy['Mô hình']
huy['objective'] = np.where(huy['model'].eq('Hybrid ARIMA-GRU'), 'residual/hybrid', 'direct')
huy['feature_set'] = huy['FS']
huy['ratio'] = huy['Split'].str.extract(r'([0-9.]+)')[0]
huy['scheme'] = huy['Scheme']
huy['seed'] = huy['Seed']
huy['mae'], huy['rmse'] = huy['MAE'], huy['RMSE']
huy['mape_pct'], huy['r2'] = huy['MAPE (%)'], huy['R2']
huy['source_file'] = str(p_huy)
huy['protocol_note'] = 'One-shot Holdout; Hybrid vẫn dùng Copper trong nhánh ARIMA'
sources['Huy'] = str(p_huy)
frames.append(keep_schema(huy))

# Công/TPE — kết quả Direct/Residual mới nhất của từng FS, seed 41–45.
p_tpe = sorted((ROOT / 'TPE+LSTM').glob('fs0[1-6]/results/*/*/metrics_by_run.csv'))
latest_tpe = {p.relative_to(ROOT / 'TPE+LSTM').parts[0]: p for p in
              sorted(p_tpe, key=lambda p: p.parent.name)}
assert set(latest_tpe) == {f'fs{i:02d}' for i in range(1, 7)}, 'Thiếu file metrics của Công'
tpe = pd.concat([pd.read_csv(p).assign(source_file=str(p)) for p in latest_tpe.values()], ignore_index=True)
fs_names = dict(zip([f'FS{i:02d}' for i in range(1, 7)],
                    ['FS01_CU', 'FS02_PAPER_CU', 'FS03_PAPER_EXO', 'FS04_METALS_CU', 'FS05_METALS_EXO', 'FS06_WTI_CU']))
tpe['feature_set'] = tpe['feature_set'].replace(fs_names)
tpe = tpe[tpe['method'].isin(['sa_lstm_v22', 'tpe_lstm'])].copy()
tpe['module'] = 'Công — PP2'
tpe['model'] = tpe['method'].map({'sa_lstm_v22': 'SA-LSTM per-FS', 'tpe_lstm': 'TPE-LSTM'})
tpe['objective'] = tpe['forecast_mode'].fillna('residual')
tpe['mape_pct'] = np.nan
tpe['protocol_note'] = 'One-shot Holdout; residual reconstruction uses Copper(t-1)'
sources['Công/TPE'] = [str(p) for p in latest_tpe.values()]
frames.append(keep_schema(tpe))

# Vĩ — iTransformer direct/residual; loại Naive.
p_vi = find_one(['Vi Task/outputs/tables/five_seed/job_mae_seed41_45.csv'], 'job_mae_seed41_45.csv')
vi = pd.read_csv(p_vi)
vi = vi[vi['model'].isin(['itransformer_direct', 'itransformer_residual'])].copy()
vi['module'] = 'Vĩ — PP3'
vi['objective'] = np.where(vi['model'].str.contains('residual'), 'residual', 'direct')
vi['source_file'] = str(p_vi)
vi['protocol_note'] = 'One-shot Holdout; residual reconstruction uses Copper(t-1)'
sources['Vĩ'] = str(p_vi)
frames.append(keep_schema(vi))

# Quân — benchmark có cả walk-forward; ở đây chỉ giữ hai strategy static Holdout.
p_quan = find_one(['quan task/result/metrics/benchmark_complete_metrics.csv'], 'benchmark_complete_metrics.csv')
quan = pd.read_csv(p_quan)
quan = quan[quan['strategy'].isin(['static_train_only', 'static_train_val'])].copy()
quan['module'] = 'Quân — PP4'
quan['model'] = 'LSTM ' + quan['target_mode'].str.lower()
quan['objective'] = quan['target_mode'].str.lower()
quan['feature_set'] = quan['feature_set'].replace({'FEATURES_4': 'FS02_PAPER_CU', 'FEATURES_3': 'FS03_PAPER_EXO'})
quan['ratio'] = 0.9
quan['scheme'] = quan['strategy']
quan['seed'] = 42
quan['mae'], quan['rmse'] = quan['MAE'], quan['RMSE']
quan['mape_pct'], quan['r2'] = quan['MAPE_pct'], quan['R2']
quan['source_file'] = str(p_quan)
quan['protocol_note'] = 'Static one-shot Holdout only; expanding/rolling rows excluded'
sources['Quân'] = str(p_quan)
frames.append(keep_schema(quan))

raw = pd.concat(frames, ignore_index=True)
print(json.dumps(sources, ensure_ascii=False, indent=2))
display(raw.groupby(['module', 'model'], as_index=False).size())

## 2. Ghép cặp có/không Copper và kiểm tra completeness

Ghép theo module, model, mục tiêu, tỷ lệ chia, chiến lược và seed. Với HPO theo từng bộ biến, siêu tham số giữa hai phía có thể khác nhau. `MAE_gain_pct > 0` nghĩa là **thêm Copper giúp giảm MAE**.

In [ ]:
PAIR_MAP = {
    'PAPER': ('FS02_PAPER_CU', 'FS03_PAPER_EXO'),
    'METALS': ('FS04_METALS_CU', 'FS05_METALS_EXO'),
}
KEYS = ['module', 'model', 'objective', 'ratio', 'scheme', 'seed']
METRICS = ['mae', 'rmse', 'mape_pct', 'r2']
pair_parts = []
missing_parts = []

for family, (with_fs, without_fs) in PAIR_MAP.items():
    with_cu = raw[raw['feature_set'].eq(with_fs)][KEYS + METRICS].copy()
    without_cu = raw[raw['feature_set'].eq(without_fs)][KEYS + METRICS].copy()
    assert not with_cu.duplicated(KEYS).any(), f'Trùng khóa phía có Copper: {family}'
    assert not without_cu.duplicated(KEYS).any(), f'Trùng khóa phía không Copper: {family}'
    paired = with_cu.merge(without_cu, on=KEYS, how='outer', suffixes=('_with', '_without'), indicator=True)
    if family in PAIR_MAP:
        # Official Module1 Holdout grid also identifies configurations with neither arm present.
        from itertools import product
        expected = pd.DataFrame([
            ('Khoa — Module1', model, 'residual' if 'residual' in model else 'direct', ratio, scheme, 42)
            for model, ratio, scheme in product(sorted(allowed_m1), (0.8, 0.9, 0.925, 0.95),
                                                ('expanding', 'rolling_756', 'recency_weighted'))
        ], columns=KEYS)
        absent = expected.merge(paired[KEYS], on=KEYS, how='left', indicator=True)
        absent = absent.loc[absent['_merge'].eq('left_only'), KEYS].assign(_merge='neither')
        paired = pd.concat([paired, absent], ignore_index=True)
    paired['family'] = family
    paired['with_feature_set'] = with_fs
    paired['without_feature_set'] = without_fs
    valid = paired['_merge'].eq('both') & paired[['mae_with', 'mae_without']].notna().all(axis=1)
    missing = paired.loc[~valid, KEYS + ['family', 'with_feature_set', 'without_feature_set', '_merge']].copy()
    missing['missing_side'] = missing.pop('_merge').map({
        'left_only': 'without_copper', 'right_only': 'with_copper',
        'neither': 'both', 'both': 'missing_mae'})
    missing_parts.append(missing)
    paired = paired.loc[valid].drop(columns='_merge').copy()
    paired['mae_gain_pct'] = 100 * (paired['mae_without'] - paired['mae_with']) / paired['mae_without']
    paired['rmse_gain_pct'] = 100 * (paired['rmse_without'] - paired['rmse_with']) / paired['rmse_without']
    paired['mape_gain_pct'] = 100 * (paired['mape_pct_without'] - paired['mape_pct_with']) / paired['mape_pct_without']
    paired['r2_gain'] = paired['r2_with'] - paired['r2_without']
    pair_parts.append(paired)

pairs = pd.concat(pair_parts, ignore_index=True)
missing_pairs = pd.concat(missing_parts, ignore_index=True)
actual_by_module = pairs.groupby('module').size().to_dict()
assert not pairs.duplicated(KEYS + ['family']).any()
assert len(pairs) > 0, 'Không có cặp Holdout hợp lệ'
display(pairs.groupby(['module', 'family']).size().rename('paired'))
display(missing_pairs.groupby(['module', 'family', 'missing_side']).size().rename('missing'))

## 3. Bảng kết quả và kiểm định mô tả

Các split test lồng nhau và các seed/config không hoàn toàn độc lập. Wilcoxon và bootstrap dưới đây là bằng chứng mô tả, không phải khẳng định nhân quả tuyệt đối.

In [ ]:
summary = (
    pairs.groupby(['module', 'model', 'objective', 'family'], as_index=False)
    .agg(
        n=('mae_gain_pct', 'size'),
        mae_with=('mae_with', 'mean'),
        mae_without=('mae_without', 'mean'),
        mean_mae_gain_pct=('mae_gain_pct', 'mean'),
        median_mae_gain_pct=('mae_gain_pct', 'median'),
        win_rate_pct=('mae_gain_pct', lambda x: 100 * np.mean(x > 0)),
        mean_rmse_gain_pct=('rmse_gain_pct', 'mean'),
        mean_r2_gain=('r2_gain', 'mean'),
    )
)

rng = np.random.default_rng(2026)
stat_rows = []
for key, part in pairs.groupby(['module', 'model', 'objective', 'family'], sort=False):
    gains = part['mae_gain_pct'].dropna().to_numpy()
    boot = np.array([rng.choice(gains, len(gains), replace=True).mean() for _ in range(5000)])
    diff = (part['mae_without'] - part['mae_with']).dropna().to_numpy()
    nonzero = diff[~np.isclose(diff, 0)]
    p_value = wilcoxon(nonzero).pvalue if len(nonzero) >= 2 else np.nan
    stat_rows.append(dict(zip(['module', 'model', 'objective', 'family'], key)) | {
        'n': len(gains), 'mean_mae_gain_pct': gains.mean(),
        'bootstrap_ci_low': np.quantile(boot, 0.025),
        'bootstrap_ci_high': np.quantile(boot, 0.975),
        'wilcoxon_p_raw': p_value,
    })
stats = pd.DataFrame(stat_rows)
stats['wilcoxon_p_holm'] = np.nan
valid = stats['wilcoxon_p_raw'].notna()
if valid.any():
    order = stats.loc[valid, 'wilcoxon_p_raw'].sort_values().index
    m = len(order)
    adjusted = np.maximum.accumulate([min(1.0, stats.loc[idx, 'wilcoxon_p_raw'] * (m - rank)) for rank, idx in enumerate(order)])
    stats.loc[order, 'wilcoxon_p_holm'] = adjusted

overall = (
    pairs.groupby(['module', 'model', 'objective'], as_index=False)
    .agg(n=('mae_gain_pct', 'size'), mean_mae_gain_pct=('mae_gain_pct', 'mean'),
         median_mae_gain_pct=('mae_gain_pct', 'median'),
         win_rate_pct=('mae_gain_pct', lambda x: 100 * np.mean(x > 0)))
    .sort_values('mean_mae_gain_pct', ascending=False)
)
display(overall.round(4))
display(summary.sort_values(['module', 'model', 'family']).round(4))

## 4. Biểu đồ và gói kết quả cho báo cáo

In [ ]:
plot_df = summary.copy()
plot_df['label'] = plot_df['module'] + ' · ' + plot_df['model']
heat = plot_df.pivot_table(index='label', columns='family', values='mean_mae_gain_pct')
fig, ax = plt.subplots(figsize=(9, max(7, 0.42 * len(heat))))
sns.heatmap(heat, annot=True, fmt='.2f', center=0, cmap='RdYlGn', linewidths=.5, ax=ax, cbar_kws={'label': 'MAE gain (%)'})
ax.set(title='Ảnh hưởng của kênh Copper — dương là giảm MAE', xlabel='Cặp feature set', ylabel='Module · Model')
fig.tight_layout()
fig.savefig(OUT / 'fig_copper_input_mae_gain_heatmap.png', dpi=180, bbox_inches='tight')
plt.show()

bar = overall.copy()
bar['label'] = bar['module'] + ' · ' + bar['model']
bar = bar.sort_values('mean_mae_gain_pct')
fig, ax = plt.subplots(figsize=(10, max(7, 0.42 * len(bar))))
colors = np.where(bar['mean_mae_gain_pct'] >= 0, '#2ca25f', '#de2d26')
ax.barh(bar['label'], bar['mean_mae_gain_pct'], color=colors)
ax.axvline(0, color='black', linewidth=1)
ax.set(title='MAE gain trung bình khi thêm Copper (hai họ feature)', xlabel='MAE gain (%)', ylabel='')
fig.tight_layout()
fig.savefig(OUT / 'fig_copper_input_overall_gain.png', dpi=180, bbox_inches='tight')
plt.show()

raw.to_csv(OUT / 'normalized_holdout_results_all_modules.csv', index=False)
pairs.to_csv(OUT / 'copper_input_paired_configs.csv', index=False)
missing_pairs.to_csv(OUT / 'copper_input_missing_pairs.csv', index=False)
summary.to_csv(OUT / 'copper_input_summary_by_family.csv', index=False)
overall.to_csv(OUT / 'copper_input_summary_overall.csv', index=False)
stats.to_csv(OUT / 'copper_input_bootstrap_wilcoxon.csv', index=False)

manifest = {
    'protocol': 'one-shot Holdout only',
    'paired_config_count': int(len(pairs)),
    'paired_by_module': actual_by_module,
    'missing_pair_count': int(len(missing_pairs)),
    'coverage_scope': 'Official Module1 grid and observed keys from the other modules',
    'tpe_sa_scope': 'Feature ablation under independently tuned hyperparameters per feature set',
    'sources': sources,
    'excluded': ['CNN', 'Naive', 'standalone ARIMA baseline', 'walk-forward rows'],
    'interpretation': 'Positive gain means adding Copper to the network input reduced error.',
    'residual_caveat': 'No-Copper variants may still use Copper(t-1) for level reconstruction; Hybrid uses Copper in ARIMA branch.',
}
(OUT / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

zip_path = OUT.parent / '05_copper_input_ablation.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUT.rglob('*')):
        if path.is_file():
            zf.write(path, path.relative_to(OUT.parent))
with zipfile.ZipFile(zip_path) as zf:
    required = ['copper_input_paired_configs.csv', 'copper_input_missing_pairs.csv', 'manifest.json']
    assert {f'{OUT.name}/{name}' for name in required} <= set(zf.namelist())
print(f'Đã lưu báo cáo: {OUT.resolve()}')
print(f'Đã đóng gói: {zip_path.resolve()}')